In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize


# =========================================================
# 1. Load data
# =========================================================
mutualfunds_path = "mutualfunds.csv"
mktetf_path = "mktetf.csv"
output_path = "q4_style_benchmark_output.xlsx"

fund_df = pd.read_csv(mutualfunds_path)
etf_df = pd.read_csv(mktetf_path)

# Standardize date columns
fund_df.columns = [c.strip() for c in fund_df.columns]
etf_df.columns = [c.strip() for c in etf_df.columns]

# Try to detect date column names
fund_date_col = "date" if "date" in [c.lower() for c in fund_df.columns] else fund_df.columns[0]
etf_date_col = "Date" if "Date" in etf_df.columns else etf_df.columns[0]

if fund_date_col != "date":
    fund_df = fund_df.rename(columns={fund_date_col: "date"})
if etf_date_col != "date":
    etf_df = etf_df.rename(columns={etf_date_col: "date"})

fund_df["date"] = pd.to_datetime(fund_df["date"])
etf_df["date"] = pd.to_datetime(etf_df["date"])

# Merge
df = pd.merge(fund_df, etf_df, on="date", how="inner").sort_values("date").reset_index(drop=True)

In [2]:
df.head()

,date,LCGFX,MRLIX,mktvw,rf,EEM,EFA,IWD,IWF,IWN,IWO,LQD,SHY,TLT
0,2016-01-29,-0.061437,-0.068841,-0.057090,0.0001,-0.050326,-0.055177,-0.052831,-0.056795,-0.065477,-0.105974,0.001228,0.006520,0.055731
1,2016-02-29,-0.015106,0.009728,0.000620,0.0002,-0.008178,-0.033345,0.000000,0.000000,0.006052,-0.008432,0.010528,0.001121,0.030866
2,2016-03-31,0.066462,0.067437,0.070506,0.0002,0.129617,0.065821,0.072667,0.067362,0.082707,0.076195,0.036162,0.001376,-0.000896
3,2016-04-29,-0.003835,-0.016245,0.011719,0.0001,0.004088,0.022218,0.021051,-0.009220,0.021357,0.011010,0.015539,0.000361,-0.007368
4,2016-05-31,0.028874,0.018349,0.014341,0.0001,-0.036929,-0.000856,0.014669,0.019624,0.017863,0.027672,-0.005124,-0.001188,0.008077


In [3]:
# =========================================================
# 2. Define ETF columns used in style analysis
# =========================================================
# Based on assignment / style analysis template
etf_cols = ["IWF", "IWD", "IWO", "IWN", "TLT", "SHY", "LQD", "EFA", "EEM"]

# Check columns exist
missing_cols = [c for c in etf_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing ETF columns in merged data: {missing_cols}")

fund_cols = ["LCGFX", "MRLIX"]
missing_funds = [c for c in fund_cols if c not in df.columns]
if missing_funds:
    raise ValueError(f"Missing fund columns in merged data: {missing_funds}")

In [51]:
# =========================================================
# 3. Optimization function
# =========================================================
def estimate_style_weights(y, X):
    """
    Minimize residual variance (equivalently SSE for fixed sample size)
    subject to:
      - weights >= 0
      - weights <= 1
      - sum(weights) = 1
    """
    n_assets = X.shape[1]

    def objective(w):
        residuals = y - X @ w
        return np.var(residuals, ddof=0)

    x0 = np.full(n_assets, 1.0 / n_assets)

    bounds = [(0.0, 1.0) for _ in range(n_assets)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(
        objective, 
        x0, 
        method='SLSQP', 
        bounds=bounds, 
        constraints=constraints,
        options={'ftol': 1e-15, 'maxiter': 1000})

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    return result.x

In [52]:
# =========================================================
# 4. Rolling benchmark construction
# =========================================================
def build_style_benchmark(data, fund_col, etf_cols, start_date="2021-01-01", end_date="2025-12-31", window=36):
    """
    For each month t in [start_date, end_date]:
      - use previous 36 months to estimate style weights
      - use month t ETF returns and estimated weights to compute benchmark return
      - save exposures, benchmark return, and fund return
    """
    data = data.sort_values("date").reset_index(drop=True)

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    results = []

    for i in range(window, len(data)):
        current_date = data.loc[i, "date"]

        if current_date < start_date or current_date > end_date:
            continue

        # previous 36 months
        train = data.iloc[i - window:i].copy()

        y = train[fund_col].to_numpy(dtype=float)
        X = train[etf_cols].to_numpy(dtype=float)

        # estimate weights
        weights = estimate_style_weights(y, X)

        # month t benchmark return
        current_etf_returns = data.loc[i, etf_cols].to_numpy(dtype=float)
        benchmark_return = float(np.dot(weights, current_etf_returns))
        fund_return = float(data.loc[i, fund_col])

        row = {
            "date": current_date,
            "fund": fund_col,
            "benchmark_return": benchmark_return,
            "fund_return": fund_return,
            "active_return": fund_return - benchmark_return
        }

        for col, w in zip(etf_cols, weights):
            row[col] = w

        results.append(row)

    return pd.DataFrame(results)

In [53]:
# =========================================================
# 5. Run for both funds
# =========================================================
lcgfx_out = build_style_benchmark(df, "LCGFX", etf_cols)
mrlix_out = build_style_benchmark(df, "MRLIX", etf_cols)

In [54]:
# =========================================================
# 6. Save output
# =========================================================
# Excel output (combined)
with pd.ExcelWriter("q4_style_benchmark_output.xlsx", engine="openpyxl") as writer:
    lcgfx_out.to_excel(writer, sheet_name="LCGFX_Q4a", index=False)
    mrlix_out.to_excel(writer, sheet_name="MRLIX_Q4a", index=False)

# CSV outputs (separate)
lcgfx_out.to_csv("q4a_LCGFX.csv", index=False)
mrlix_out.to_csv("q4a_MRLIX.csv", index=False)

print("Done.")
print(f"Saved output to: {output_path}")

Done.
Saved output to: q4_style_benchmark_output.xlsx


In [55]:
# =========================================================
# 7. Optional: print Q4(b) summary stats
# =========================================================
def summarize_ir(df_out):
    avg_active_return = df_out["active_return"].mean()
    tracking_error = df_out["active_return"].std(ddof=1)
    information_ratio = avg_active_return / tracking_error if tracking_error != 0 else np.nan
    return avg_active_return, tracking_error, information_ratio

for name, out in [("LCGFX", lcgfx_out), ("MRLIX", mrlix_out)]:
    avg_ar, te, ir = summarize_ir(out)
    print(f"\n{name}")
    print(f"Average active return: {avg_ar:.6f}")
    print(f"Tracking error:      {te:.6f}")
    print(f"Information ratio:   {ir:.6f}")


LCGFX
Average active return: -0.002132
Tracking error:      0.013550
Information ratio:   -0.157320

MRLIX
Average active return: 0.000203
Tracking error:      0.013437
Information ratio:   0.015078


In [57]:
benchmark_df = pd.read_csv("q4a_LCGFX.csv")
benchmark_df

,date,fund,benchmark_return,fund_return,active_return,IWF,IWD,IWO,IWN,TLT,SHY,LQD,EFA,EEM
0,2021-01-29,LCGFX,-0.008827,-0.031470,-0.022643,0.785858,9.704996e-02,2.078655e-02,0.000000e+00,6.872262e-02,0.000000e+00,0.000000e+00,2.758331e-02,0.000000e+00
1,2021-02-26,LCGFX,0.003436,0.043162,0.039726,0.785543,1.360911e-01,2.013574e-17,7.017374e-18,7.836635e-02,4.770622e-18,5.641804e-18,2.749388e-17,0.000000e+00
2,2021-03-31,LCGFX,0.022532,0.011158,-0.011374,0.697261,2.319584e-01,0.000000e+00,2.112982e-17,7.078079e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,2021-04-30,LCGFX,0.059475,0.068506,0.009031,0.721434,2.028678e-01,2.923270e-03,2.267644e-17,7.277543e-02,0.000000e+00,4.665218e-18,7.014304e-18,0.000000e+00
4,2021-05-28,LCGFX,-0.005674,-0.001721,0.003953,0.731680,1.965098e-01,0.000000e+00,0.000000e+00,7.180993e-02,1.783676e-18,2.146415e-18,3.790513e-18,0.000000e+00
5,2021-06-30,LCGFX,0.045468,0.049138,0.003670,0.730647,1.974182e-01,0.000000e+00,3.657129e-17,7.193466e-02,6.994794e-18,0.000000e+00,0.000000e+00,0.000000e+00
6,2021-07-30,LCGFX,0.029084,0.036976,0.007892,0.731489,1.965096e-01,0.000000e+00,1.766187e-18,7.200104e-02,9.381666e-18,0.000000e+00,1.997953e-17,2.569261e-17
7,2021-08-31,LCGFX,0.030258,0.027338,-0.002920,0.728539,2.009091e-01,1.546477e-17,0.000000e+00,7.055224e-02,0.000000e+00,0.000000e+00,1.687342e-17,8.293853e-20
8,2021-09-30,LCGFX,-0.050335,-0.051292,-0.000957,0.730009,1.991262e-01,3.860273e-17,0.000000e+00,7.086443e-02,0.000000e+00,0.000000e+00,1.259619e-17,1.048026e-17
9,2021-10-29,LCGFX,0.075980,0.090650,0.014670,0.734671,1.953368e-01,9.332587e-18,0.000000e+00,6.999247e-02,0.000000e+00,5.559604e-18,1.038209e-17,4.744267e-18
